
# RuCoLA: классификация
Цель: применить трансформеры к задаче RuCoLA.
Состав:
1. Загрузка `in_domain_train.csv` и `in_domain_dev.csv` через `huggingface_hub` без `datasets`.
2. Разбиение на `train` и `val` из `in_domain_train`.
3. Дообучение RuBERT простым циклом обучения и оценка.
4. Zero- / few-shot с RuGPT3 (k ∈ {0,1,2,4}).
5. Дообучение RuT5 простым циклом обучения и оценка.
6. Сводная таблица метрик.


In [1]:

# Установка минимальных зависимостей. Без datasets -> не будет конфликтов с pyarrow/RAPIDS.
!pip -q install "transformers>=4.40.0" "accelerate>=0.28.0" "sentencepiece>=0.1.99" "huggingface_hub>=0.23.0" scikit-learn -U


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 49.0 MB/s eta 0:00:00


In [2]:

import os, io, math, random, time
import numpy as np
import pandas as pd
from pathlib import Path

import torch
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

from huggingface_hub import hf_hub_download
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, AutoModelForCausalLM,
                          T5ForConditionalGeneration, T5Tokenizer, get_linear_schedule_with_warmup)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
torch.backends.cudnn.benchmark = True
seed = 42
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


Device: cuda



## Загрузка данных RuCoLA
Берем файлы из датасета `RussianNLP/rucola`

In [3]:

repo_id = "RussianNLP/rucola"
train_path = hf_hub_download(repo_id=repo_id, repo_type="dataset", filename="data/in_domain_train.csv")
dev_path   = hf_hub_download(repo_id=repo_id, repo_type="dataset", filename="data/in_domain_dev.csv")

df_train_full = pd.read_csv(train_path)
df_dev        = pd.read_csv(dev_path)

print("Train full shape:", df_train_full.shape, "Dev shape:", df_dev.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


in_domain_train.csv: 0.00B [00:00, ?B/s]

in_domain_dev.csv: 0.00B [00:00, ?B/s]

Train full shape: (7869, 5) Dev shape: (983, 5)


In [4]:
# parser для RuCoLA
import pandas as pd
from sklearn.model_selection import train_test_split

def pick_text_label_cols(df):
    cols_lower = {c.lower(): c for c in df.columns}
    # текст
    for cand in ["sentence", "text"]:
        if cand in cols_lower:
            text_col = cols_lower[cand]
            break
    else:
        raise ValueError(f"Не найден текстовый столбец среди {list(df.columns)}")

    # метка
    for cand in ["label", "acceptable", "gold_label"]:
        if cand in cols_lower:
            label_col = cols_lower[cand]
            break
    else:
        raise ValueError(f"Не найден столбец метки среди {list(df.columns)}")

    return text_col, label_col

def normalize_labels(series):
    mapped = []
    for v in series:
        s = str(v).strip().lower()
        if s in {"1", "acceptable", "true", "yes"}:
            mapped.append(1)
        elif s in {"0", "unacceptable", "false", "no"}:
            mapped.append(0)
        else:
            try:
                mapped.append(1 if int(s) == 1 else 0)
            except:
                mapped.append(0)
    return pd.Series(mapped, dtype=int)

# 1) выбираем корректные колонки
TEXT_COL_TRAIN, LABEL_COL_TRAIN = pick_text_label_cols(df_train_full)
TEXT_COL_DEV,   LABEL_COL_DEV   = pick_text_label_cols(df_dev)

# 2) нормализуем метки
df_train_full = df_train_full.copy()
df_dev = df_dev.copy()
df_train_full["label"] = normalize_labels(df_train_full[LABEL_COL_TRAIN])
df_dev["label"]        = normalize_labels(df_dev[LABEL_COL_DEV])

# 3) проверим распределение
print("Train label value_counts():")
print(df_train_full["label"].value_counts(dropna=False))
print("\nDev label value_counts():")
print(df_dev["label"].value_counts(dropna=False))

# 4) строим выборки
train_df, val_df = train_test_split(
    df_train_full[[TEXT_COL_TRAIN, "label"]],
    test_size=0.1,
    random_state=42,
    stratify=df_train_full["label"]
)
train_df = train_df.reset_index(drop=True).rename(columns={TEXT_COL_TRAIN: "text"})
val_df   = val_df.reset_index(drop=True).rename(columns={TEXT_COL_TRAIN: "text"})
test_df  = df_dev[[TEXT_COL_DEV, "label"]].reset_index(drop=True).rename(columns={TEXT_COL_DEV: "text"})

print("\nSplits ->", "train:", train_df.shape, "val:", val_df.shape, "test:", test_df.shape)

# 5) sanity-check: покажем несколько положительных примеров
pos_samples = train_df[train_df["label"] == 1].head(5)
neg_samples = train_df[train_df["label"] == 0].head(5)
print("\nПервые 5 положительных примеров:")
for i, row in pos_samples.iterrows():
    print("-", row["text"])
print("\nПервые 5 отрицательных примеров:")
for i, row in neg_samples.iterrows():
    print("-", row["text"])


Train label value_counts():
label
1    5864
0    2005
Name: count, dtype: int64

Dev label value_counts():
label
1    733
0    250
Name: count, dtype: int64

Splits -> train: (7082, 2) val: (787, 2) test: (983, 2)

Первые 5 положительных примеров:
- Я не знаю Машу Трофимову.
- Она, конечно, знала, что рискует, отказываясь от его предложений.
- Я всегда получаю удовольствие, когда читаю его работы.
- После новогодних поздравлений началась раздача подарков.
- Вынь у меня из кармана ключ, он мне мешает.

Первые 5 отрицательных примеров:
- Костюм не был в химчистке весь год.
- Она неправильно приняла возражения оппонента.
- Телега сломана два дня назад, но сегодня ее починили.
- Я имею книгу, чтобы читать.
- Посетители подолгу рассматривали и восхищались полотнами художников-передвижников.



## Классификация: RuBERT, простой цикл обучения
Модель по умолчанию `DeepPavlov/rubert-base-cased`.


In [5]:

model_name_cls = "DeepPavlov/rubert-base-cased"
tokenizer_cls = AutoTokenizer.from_pretrained(model_name_cls, use_fast=True)
model_cls = AutoModelForSequenceClassification.from_pretrained(model_name_cls, num_labels=2).to(device)

class TextClsDataset(Dataset):
    def __init__(self, df):
        self.texts = df["text"].tolist()
        self.labels = df["label"].tolist()
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        return {"text": self.texts[idx], "labels": int(self.labels[idx])}

def collate_cls(batch):
    texts = [b["text"] for b in batch]
    labels = torch.tensor([b["labels"] for b in batch], dtype=torch.long)
    enc = tokenizer_cls(texts, truncation=True, padding=True, return_tensors="pt")
    enc["labels"] = labels
    return {k: v.to(device) for k, v in enc.items()}

train_loader = DataLoader(TextClsDataset(train_df), batch_size=16, shuffle=True, collate_fn=collate_cls)
val_loader   = DataLoader(TextClsDataset(val_df),   batch_size=32, shuffle=False, collate_fn=collate_cls)
test_loader  = DataLoader(TextClsDataset(test_df),  batch_size=32, shuffle=False, collate_fn=collate_cls)

optimizer = torch.optim.AdamW(model_cls.parameters(), lr=2e-5)
epochs = 5
num_training_steps = epochs * len(train_loader)
warmup_steps = max(0, int(0.06 * num_training_steps))
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=num_training_steps)
scaler = GradScaler(enabled=(device=="cuda"))

def evaluate_cls(loader):
    model_cls.eval()
    preds_all, labels_all = [], []
    with torch.no_grad():
        for batch in loader:
            with autocast(enabled=(device=="cuda")):
                out = model_cls(**batch)
                logits = out.logits
            preds = logits.argmax(dim=-1).detach().cpu().numpy().tolist()
            labels = batch["labels"].detach().cpu().numpy().tolist()
            preds_all.extend(preds); labels_all.extend(labels)
    acc = accuracy_score(labels_all, preds_all)
    f1  = f1_score(labels_all, preds_all)
    return {"accuracy": acc, "f1": f1}

for ep in range(1, epochs+1):
    model_cls.train()
    running = 0.0
    for step, batch in enumerate(train_loader, 1):
        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=(device=="cuda")):
            out = model_cls(**batch)
            loss = out.loss
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        running += loss.item()
        if step % 50 == 0:
            print(f"epoch {ep} step {step}/{len(train_loader)} loss {running/step:.4f}")
    val_metrics = evaluate_cls(val_loader)
    print(f"epoch {ep} val: acc={val_metrics['accuracy']:.4f} f1={val_metrics['f1']:.4f}")

test_metrics_cls = evaluate_cls(test_loader)
print("RuBERT test:", test_metrics_cls)


tokenizer_config.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/714M [00:00<?, ?B/s]

...

epoch 5 step 50/443 loss 0.0758
epoch 5 step 100/443 loss 0.0674
epoch 5 step 150/443 loss 0.0701
epoch 5 step 200/443 loss 0.0648
epoch 5 step 250/443 loss 0.0626
epoch 5 step 300/443 loss 0.0637
epoch 5 step 350/443 loss 0.0625
epoch 5 step 400/443 loss 0.0640


/tmp/ipython-input-3779210783.py:37: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(device=="cuda")):


epoch 5 val: acc=0.7853 f1=0.8636


/tmp/ipython-input-3779210783.py:37: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(device=="cuda")):


RuBERT test: {'accuracy': 0.8046795523906409, 'f1': 0.8786346396965866}


In [6]:
# Демонстрация работы классификатора
import torch

def classify_sentence(sentence: str):
    model_cls.eval()
    inputs = tokenizer_cls(sentence, return_tensors="pt", truncation=True, padding=True).to(device)
    with torch.no_grad():
        logits = model_cls(**inputs).logits
        pred = torch.argmax(logits, dim=-1).item()
    label = "приемлемо (1)" if pred == 1 else "неприемлемо (0)"
    return label

# Пример
example = "Мама мыла раму."
print(example, "->", classify_sentence(example))

# Можно вводить произвольный текст:
while True:
    s = input("Введите предложение (или пустую строку для выхода): ").strip()
    if not s:
        break
    print("Класс:", classify_sentence(s))

Мама мыла раму. -> приемлемо (1)
Введите предложение (или пустую строку для выхода): ничаго нипанятна
Класс: неприемлемо (0)
Введите предложение (или пустую строку для выхода): Здравствуйте, я ваша тётя
Класс: приемлемо (1)
Введите предложение (или пустую строку для выхода): Нейрохирург завтра делал операцию
Класс: приемлемо (1)
Введите предложение (или пустую строку для выхода): Пистолет в облаках танцует чачу
Класс: приемлемо (1)
Введите предложение (или пустую строку для выхода): много ощибок в предложени
Класс: неприемлемо (0)
Введите предложение (или пустую строку для выхода): 



## Zero-/Few-shot с RuGPT3
Генерация с шаблонами.


In [7]:

gpt_name = "sberbank-ai/rugpt3medium_based_on_gpt2"  # medium безопаснее по памяти
tok_gpt = AutoTokenizer.from_pretrained(gpt_name)
mdl_gpt = AutoModelForCausalLM.from_pretrained(gpt_name).to(device)

def make_prompt(examples, query_sentence, template_id=0):
    if template_id == 0:
        head = "Определи приемлемость русского предложения. Ответ 1 если приемлемо, 0 если нет."
        fmt = "Предложение: {s} Ответ: {y}"
    elif template_id == 1:
        head = "Классификация приемлемости. 1=приемлемо, 0=неприемлемо."
        fmt = "Текст: {s} Метка: {y}"
    else:
        head = "Напиши 1, если предложение грамматически корректно, иначе 0."
        fmt = "Вход: {s} Выход: {y}"
    body = "".join(fmt.format(s=s, y=y) for s, y in examples)
    body += fmt.format(s=query_sentence, y="")
    return head + body

def parse_label(text):
    t = text.strip().splitlines()[-1].strip()
    for ch in t:
        if ch in ["0","1"]:
            return int(ch)
    if "не" in t.lower():
        return 0
    return 1

def sample_k_examples(df, k):
    if k == 0: return []
    pos = df[df["label"]==1]
    neg = df[df["label"]==0]
    ex = []
    m = min(len(pos), len(neg), max(1, k//2))
    ex += list(zip(pos.sample(m, random_state=42)["text"], [1]*m))
    ex += list(zip(neg.sample(m, random_state=42)["text"], [0]*m))
    while len(ex) < k:
        row = df.sample(1, random_state=42+len(ex)).iloc[0]
        ex.append((row["text"], int(row["label"])))
    return ex[:k]

Ks = [0,1,2,4]
template_ids = [0,1,2]
N_EVAL_LIMIT = 200

results_gpt = []
test_pairs = list(zip(test_df["text"].tolist(), test_df["label"].tolist()))
if N_EVAL_LIMIT is not None:
    test_pairs = test_pairs[:N_EVAL_LIMIT]

for k in Ks:
    examples = sample_k_examples(train_df, k)
    for tpl in template_ids:
        preds, golds = [], []
        for s, y in test_pairs:
            prompt = make_prompt(examples, s, template_id=tpl)
            input_ids = tok_gpt.encode(prompt, return_tensors="pt").to(device)
            with torch.no_grad():
                out = mdl_gpt.generate(
                    input_ids,
                    max_new_tokens=8,
                    do_sample=False,
                    num_beams=1,
                    pad_token_id=tok_gpt.eos_token_id
                )
            gen = tok_gpt.decode(out[0][input_ids.shape[-1]:], skip_special_tokens=True)
            preds.append(parse_label(gen)); golds.append(int(y))
        acc = accuracy_score(golds, preds); f1 = f1_score(golds, preds)
        results_gpt.append({"method": f"RuGPT3 k={k} tpl={tpl}", "accuracy": acc, "f1": f1})
        print(f"k={k} tpl={tpl} -> acc={acc:.4f} f1={f1:.4f}")

gpt_table = pd.DataFrame(results_gpt)
gpt_table.head()


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/574 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/761 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.73G [00:00<?, ?B/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


k=0 tpl=0 -> acc=0.7150 f1=0.8278
k=0 tpl=1 -> acc=0.7400 f1=0.8497
k=0 tpl=2 -> acc=0.5650 f1=0.7051
k=1 tpl=0 -> acc=0.7600 f1=0.8636
k=1 tpl=1 -> acc=0.7600 f1=0.8636
k=1 tpl=2 -> acc=0.7550 f1=0.8604
k=2 tpl=0 -> acc=0.2400 f1=0.0380
k=2 tpl=1 -> acc=0.4350 f1=0.5066
k=2 tpl=2 -> acc=0.2400 f1=0.0256
k=4 tpl=0 -> acc=0.2350 f1=0.0000
k=4 tpl=1 -> acc=0.2450 f1=0.0131
k=4 tpl=2 -> acc=0.2450 f1=0.0131


,method,accuracy,f1
0,RuGPT3 k=0 tpl=0,0.715,0.827795
1,RuGPT3 k=0 tpl=1,0.740,0.849711
2,RuGPT3 k=0 tpl=2,0.565,0.705085
3,RuGPT3 k=1 tpl=0,0.760,0.863636
4,RuGPT3 k=1 tpl=1,0.760,0.863636



## RuT5: простой цикл обучения
Вход: предложение. Выход: символ 0 или 1.


In [16]:
# Балансировка train и подготовка даталоадеров для RuT5

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import T5Tokenizer

# Балансировка классов
min_count = train_df["label"].value_counts().min()
train_bal = pd.concat([
    train_df[train_df["label"] == 0].sample(min_count, random_state=42),
    train_df[train_df["label"] == 1].sample(min_count, random_state=42),
]).sample(frac=1, random_state=42).reset_index(drop=True)

print("Balanced train size:", train_bal.shape)
print("Balanced distribution:", train_bal["label"].value_counts())

# Жесткий промпт для лучшей дифференциации
PROMPT_PREFIX = "Определи, является ли предложение грамматически корректным. Ответ 1 если корректно, иначе 0: "

t5_name = "ai-forever/ruT5-base"
tok_t5 = T5Tokenizer.from_pretrained(t5_name)

class T5Seq2SeqDataset(Dataset):
    def __init__(self, df):
        self.inputs = [PROMPT_PREFIX + t for t in df["text"].tolist()]
        self.targets = [str(int(y)) for y in df["label"].tolist()]
    def __len__(self):
        return len(self.inputs)
    def __getitem__(self, idx):
        return {"input_text": self.inputs[idx], "target_text": self.targets[idx]}

def collate_t5(batch):
    inputs = [b["input_text"] for b in batch]
    targets = [b["target_text"] for b in batch]
    enc = tok_t5(inputs, max_length=128, truncation=True, padding=True, return_tensors="pt")
    with tok_t5.as_target_tokenizer():
        lab = tok_t5(targets, max_length=4, truncation=True, padding=True, return_tensors="pt")
    enc["labels"] = lab["input_ids"]
    return {k: v.to(device) for k, v in enc.items()}

train_t5_loader = DataLoader(T5Seq2SeqDataset(train_bal), batch_size=16, shuffle=True,  collate_fn=collate_t5)
val_t5_loader   = DataLoader(T5Seq2SeqDataset(val_df),   batch_size=32, shuffle=False, collate_fn=collate_t5)
test_t5_loader  = DataLoader(T5Seq2SeqDataset(test_df),  batch_size=32, shuffle=False, collate_fn=collate_t5)


Balanced train size: (3608, 2)
Balanced distribution: label
1    1804
0    1804
Name: count, dtype: int64


In [19]:
# Обучение RuT5 простым циклом с AMP, шедулером и weight decay

import random
from sklearn.metrics import accuracy_score, f1_score
from transformers import T5ForConditionalGeneration, get_linear_schedule_with_warmup
from torch.cuda.amp import autocast, GradScaler

seed = 42
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed);
if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

mdl_t5 = T5ForConditionalGeneration.from_pretrained(t5_name).to(device)

optimizer = torch.optim.AdamW(mdl_t5.parameters(), lr=1e-4, weight_decay=0.01)
epochs_t5 = 6
num_training_steps = epochs_t5 * len(train_t5_loader)
warmup_steps = max(0, int(0.06 * num_training_steps))
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=num_training_steps)
scaler = GradScaler(enabled=(device == "cuda"))

def eval_t5(loader):
    mdl_t5.eval()
    preds_all, labels_all = [], []
    with torch.no_grad():
        for batch in loader:
            with autocast(enabled=(device == "cuda")):
                out_ids = mdl_t5.generate(
                    input_ids=batch["input_ids"],
                    attention_mask=batch["attention_mask"],
                    max_new_tokens=4
                )
            pred_texts = tok_t5.batch_decode(out_ids, skip_special_tokens=True)
            p = [1 if (len(s.strip()) > 0 and s.strip()[0] == "1") else 0 for s in pred_texts]

            labs = batch["labels"].detach().cpu().numpy()
            labs = np.where(labs != -100, labs, tok_t5.pad_token_id)
            lab_texts = tok_t5.batch_decode(labs, skip_special_tokens=True)
            r = [1 if (len(s.strip()) > 0 and s.strip()[0] == "1") else 0 for s in lab_texts]

            preds_all.extend(p)
            labels_all.extend(r)
    return {"accuracy": accuracy_score(labels_all, preds_all),
            "f1": f1_score(labels_all, preds_all)}

for ep in range(1, epochs_t5 + 1):
    mdl_t5.train()
    running = 0.0
    for step, batch in enumerate(train_t5_loader, 1):
        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=(device == "cuda")):
            out = mdl_t5(**batch)
            loss = out.loss
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        running += float(loss)
        if step % 50 == 0:
            print(f"epoch {ep} step {step}/{len(train_t5_loader)} loss {running/step:.4f}")
    vm = eval_t5(val_t5_loader)
    print(f"epoch {ep} val: acc={vm['accuracy']:.4f} f1={vm['f1']:.4f}")

test_metrics_t5 = eval_t5(test_t5_loader)
print("RuT5 test:", test_metrics_t5)

# Сохранение чекпойнта после обучения
mdl_t5.save_pretrained("./rut5_rucola_balanced_ckpt")
tok_t5.save_pretrained("./rut5_rucola_balanced_ckpt")


/tmp/ipython-input-1763950598.py:19: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=(device == "cuda"))
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4034: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
/tmp/ipython-input-1763950598.py:50: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(device == "cuda")):


epoch 1 step 50/226 loss 5.5658
epoch 1 step 100/226 loss 3.0131
epoch 1 step 150/226 loss 2.1404


KeyboardInterrupt: 

In [20]:
# Инференс-функция для RuT5 и проверка на нескольких предложениях

import torch

def rut5_predict(sentence: str) -> str:
    mdl_t5.eval()
    prompt = PROMPT_PREFIX + sentence
    enc = tok_t5([prompt], return_tensors="pt", truncation=True, padding=True, max_length=128).to(device)
    with torch.no_grad():
        out_ids = mdl_t5.generate(
            input_ids=enc["input_ids"],
            attention_mask=enc["attention_mask"],
            max_new_tokens=4
        )
    pred_text = tok_t5.batch_decode(out_ids, skip_special_tokens=True)[0].strip()
    label = 1 if (len(pred_text) > 0 and pred_text[0] == "1") else 0
    return "приемлемо (1)" if label == 1 else "неприемлемо (0)"

samples = [
    "Мама мыла раму.",
    "что кого куда э",
    "Зеленая мысль спит яростно."
]
for s in samples:
    print(s, "->", rut5_predict(s))


Мама мыла раму. -> приемлемо (1)
что кого куда э -> приемлемо (1)
Зеленая мысль спит яростно. -> приемлемо (1)



## Сводка результатов


In [21]:

rows = []
rows.append({"method": "RuBERT finetune", **test_metrics_cls})
if 'gpt_table' in globals():
    rows += gpt_table.to_dict("records")
rows.append({"method": "RuT5 finetune", **test_metrics_t5})
summary_df = pd.DataFrame(rows)
summary_df


,method,accuracy,f1
0,RuBERT finetune,0.804680,0.878635
1,RuGPT3 k=0 tpl=0,0.715000,0.827795
2,RuGPT3 k=0 tpl=1,0.740000,0.849711
3,RuGPT3 k=0 tpl=2,0.565000,0.705085
4,RuGPT3 k=1 tpl=0,0.760000,0.863636
5,RuGPT3 k=1 tpl=1,0.760000,0.863636
6,RuGPT3 k=1 tpl=2,0.755000,0.860399
7,RuGPT3 k=2 tpl=0,0.240000,0.037975
8,RuGPT3 k=2 tpl=1,0.435000,0.506550
9,RuGPT3 k=2 tpl=2,0.240000,0.025641


*Выводы*
- дообучил RuBert, качество на цифрах неплохое, но в тестах он умеет хорошо определять только явные ошибки. Семантически некорректные предложения он пропускает как нормальные
- zero-shot не очень хорошо себя показывает в сравнении с rubert
- rut5 обучился плохо, видимо нужно регуляризация. У него все классифицируется как приемлемый класс на текущий момент